# Fall Detection — Complete Real-Data Run (Fixed)

এই notebook-টি Save & Run All-এর জন্য তৈরি। এটি attached `fall-sim-code` এবং `fall-datasets-raw` খুঁজে real-format adapter বসায়, empty UMAFall fold-এ crash করে না, এবং শেষ পর্যন্ত reports/models/ZIP তৈরি করে।

**প্রথম run-এ smoke test**: 2 epochs। সব parser positive হলে পরে Cell 1-এর `FINAL_RUN=True` করে আবার Save & Run All করুন।

In [ ]:
# CELL 1 — setup
from pathlib import Path
import os, sys, shutil, json, re, gc, inspect, importlib, traceback, zipfile
import numpy as np, pandas as pd
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working/fall-sim')
FINAL_RUN=False
EPOCHS=100 if FINAL_RUN else 2
PATIENCE=15 if FINAL_RUN else 1
RUN_E1=True; RUN_E2=True; RUN_E3=True; RUN_E4=True; RUN_E5=True

def dirs():
    return [p for p in INPUT.rglob('*') if p.is_dir()]
code=[]
for p in INPUT.rglob('config.py'):
    if (p.parent/'src').is_dir() and (p.parent/'tools').is_dir() and 'fall-sim' in str(p).lower(): code.append(p.parent)
if not code: raise FileNotFoundError('fall-sim-code input attach করা হয়নি')
source=sorted(code,key=lambda p:len(str(p)))[0]
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(source,WORK); os.chdir(WORK); sys.path.insert(0,str(WORK))
print('Code:',source); print('Work:',WORK)
print('Inputs:',[str(p) for p in INPUT.iterdir()])


In [ ]:
# CELL 2 — locate real dataset roots

def find_root(names):
    cand=[]
    for p in INPUT.rglob('*'):
        if not p.is_dir(): continue
        s=str(p).lower()
        if any(x.lower() in s for x in names): cand.append(p)
    return sorted(cand,key=lambda p:(len(str(p)),str(p)))
roots={}
for name, markers in {'SisFall':['sisfall'],'KFall':['kfall'],'FallAllD':['fallalld'],'UMAFall':['umafall']}.items():
    cc=find_root(markers)
    if cc: roots[name]=cc[0]
    print(name, roots.get(name,'MISSING'))
if len(roots)<4: print('WARNING: missing roots:',set(['SisFall','KFall','FallAllD','UMAFall'])-set(roots))


In [ ]:
# CELL 3 — robust real-format adapter
from fractions import Fraction
from scipy.signal import resample_poly
from src import preprocessing as prep
_REAL={}
def resize(x,n):
    x=np.asarray(x,np.float32)
    if len(x)==n:return x
    q=np.linspace(0,len(x)-1,n); lo=np.floor(q).astype(int); hi=np.minimum(lo+1,len(x)-1); a=(q-lo).astype(np.float32)
    return x[lo]*(1-a[:,None])+x[hi]*a[:,None]
def rs(x,fs,target=50.):
    if len(x)<2 or abs(fs-target)<.5:return np.asarray(x,np.float32)
    f=Fraction(float(target)/float(fs)).limit_denominator(1000)
    return resample_poly(np.asarray(x,np.float32),f.numerator,f.denominator,axis=0).astype(np.float32)
def pack(parts,n=100):
    parts=[p for p in parts if p is not None and len(p['X'])]
    if not parts:return {'X':np.empty((0,n,6),np.float32),'y3':np.empty(0,np.int8),'y_post':np.empty(0,np.int8),'y_pre':np.empty(0,np.int8),'subject':np.empty(0,str),'age':np.empty(0,str),'kind':np.empty(0,str),'dataset':np.empty(0,str),'trial':np.empty(0,str),'t0':np.empty(0,np.float32),'t1':np.empty(0,np.float32),'onset_s':np.empty(0,np.float32),'impact_s':np.empty(0,np.float32),'placement':np.empty(0,str)}
    out={}
    for k in ['X','y3','y_post','y_pre','subject','age','kind','dataset','trial','t0','t1','onset_s','impact_s','placement']:
        out[k]=np.concatenate([p[k] for p in parts])
    return out
def win(data,fs,sub,kind,trial,ds,impact=-1,placement='waist'):
    data=rs(data,fs); n=100; st=25
    if len(data)<n:return None
    o={k:[] for k in ['X','y3','y_post','y_pre','subject','age','kind','dataset','trial','t0','t1','onset_s','impact_s','placement']}
    for a in range(0,len(data)-n+1,st):
        b=a+n; t0=a/50.; t1=b/50.; cls=2 if impact>=0 and t0<=impact<t1 else 0
        o['X'].append(data[a:b]); o['y3'].append(cls); o['y_post'].append(int(cls>0));o['y_pre'].append(0);o['subject'].append(str(sub));o['age'].append('unknown');o['kind'].append(str(kind));o['dataset'].append(ds);o['trial'].append(str(trial));o['t0'].append(t0);o['t1'].append(t1);o['onset_s'].append(-1.);o['impact_s'].append(impact);o['placement'].append(placement)
    for k in ['X']:o[k]=np.asarray(o[k],np.float32)
    for k in ['y3','y_post','y_pre']:o[k]=np.asarray(o[k],np.int8)
    for k in ['t0','t1','onset_s','impact_s']:o[k]=np.asarray(o[k],np.float32)
    for k in ['subject','age','kind','dataset','trial','placement']:o[k]=np.asarray(o[k],str)
    return o
def _numeric_umafall(p):
    lines=p.read_text(errors='ignore').splitlines(); waist=2
    for line in lines[:30]:
        m=re.search(r';\s*(\d+)\s*;\s*WAIST',line.upper())
        if m:waist=int(m.group(1))
    rows=[]
    for line in lines:
        if not line.strip() or line.lstrip().startswith('%'):continue
        z=[x.strip() for x in line.split(';')]
        if len(z)<7:continue
        try: rows.append([float(x.replace(',','.')) for x in z[:7]])
        except: pass
    return np.asarray(rows,np.float32),waist
def load_umafall():
    key='uma'
    if key in _REAL:return _REAL[key]
    root=roots['UMAFall']; parts=[]; files=list(root.rglob('UMAFall_Subject_*.csv'))
    for p in files:
        try:
            a,sid=_numeric_umafall(p)
            if len(a)<20:continue
            acc=a[(a[:,6]==sid)&(a[:,5]==0),2:5]; gyr=a[(a[:,6]==sid)&(a[:,5]==1),2:5]
            if len(acc)<20:continue
            if len(gyr):
                g=np.column_stack([np.interp(np.arange(len(acc)),np.linspace(0,len(acc)-1,len(gyr)),gyr[:,j]) for j in range(3)])
            else:g=np.zeros_like(acc)
            d=np.c_[acc,g]; impact=float(np.argmax(np.linalg.norm(acc,axis=1))/20.) if '_Fall_' in p.name else -1
            q=win(d,20.,re.search(r'Subject_(\d+)',p.name).group(1), 'fall' if impact>=0 else 'adl',p.stem,'UMAFall',impact)
            if q:parts.append(q)
        except Exception as e: print('skip UMA',p.name,e)
    out=pack(parts);print('UMAFall waist windows:',len(out['X']));_REAL[key]=out;return out
def load_fallalld():
    key='fall';
    if key in _REAL:return _REAL[key]
    parts=[]; root=roots['FallAllD']
    for p in root.rglob('*_A.dat'):
        m=re.search(r'S(\d+)_D(\d+)_A(\d+)_T(\d+)_A\.dat$',p.name)
        if not m:continue
        sid,dev,act,tr=map(int,m.groups());
        if dev!=3:continue
        g=p.with_name(p.name[:-6]+'_G.dat')
        if not g.exists():continue
        try:
            ac=np.loadtxt(p,delimiter=',',dtype=np.float32).reshape(-1,3)/4096.; gy=np.loadtxt(g,delimiter=',',dtype=np.float32).reshape(-1,3)/16.4; n=min(len(ac),len(gy)); d=np.c_[ac[:n],gy[:n]]
            impact=float(np.argmax(np.linalg.norm(ac[:n],axis=1))/238.) if act>=101 else -1
            q=win(d,238.,f'S{sid:02d}','fall' if impact>=0 else 'adl',p.stem,'FallAllD',impact)
            if q:parts.append(q)
        except Exception as e:print('skip FallAllD',p.name,e)
    out=pack(parts);print('FallAllD waist windows:',len(out['X']));_REAL[key]=out;return out


def load_sisfall():
    if 'sis' in _REAL:return _REAL['sis']
    root=roots['SisFall']; parts=[]
    for split in ['train','val','test']:
        xp=next(root.rglob(f'x_{split}_3'),None); yp=next(root.rglob(f'y_{split}_3'),None)
        if xp is None or yp is None: continue
        yraw=np.fromfile(str(yp),dtype=np.float32)
        n=len(yraw)//3; y=yraw[:n*3].reshape(n,3).argmax(1).astype(np.int8)
        xraw=np.fromfile(str(xp),dtype=np.float32)
        if len(xraw)%(n*6): continue
        x=xraw.reshape(n,-1,6); x=np.asarray([resize(z,100) for z in x],np.float32)
        p={'X':x,'y3':y,'y_post':(y>0).astype(np.int8),'y_pre':(y==1).astype(np.int8),'subject':np.array([f'{split}_{i%38:02d}' for i in range(n)]),'age':np.array(['unknown']*n),'kind':np.array([split]*n),'dataset':np.array(['SisFall']*n),'trial':np.array([f'{split}_{i}' for i in range(n)]),'t0':np.zeros(n,np.float32),'t1':np.ones(n,np.float32)*2,'onset_s':np.full(n,-1,np.float32),'impact_s':np.full(n,-1,np.float32),'placement':np.array(['waist']*n)}
        parts.append(p); print('SisFall',split,x.shape,np.bincount(y,minlength=3))
    out=pack(parts); print('SisFall windows:',len(out['X'])); _REAL['sis']=out; return out

def col(df, names):
    norm={re.sub(r'[^a-z0-9]','',str(c).lower()):c for c in df.columns}
    for name in names:
        k=re.sub(r'[^a-z0-9]','',name.lower())
        if k in norm:return norm[k]
    for c in df.columns:
        z=re.sub(r'[^a-z0-9]','',str(c).lower())
        if any(re.sub(r'[^a-z0-9]','',n.lower()) in z for n in names):return c
    raise KeyError((names,list(df.columns)))

def load_kfall():
    if 'kf' in _REAL:return _REAL['kf']
    root=roots['KFall']; sensor=next(root.rglob('sensor_data'),None); label=next(root.rglob('label_data'),None); parts=[]; labels={}
    if label:
        for lp in label.rglob('*_label.xlsx'):
            sub=lp.stem.replace('_label',''); df=pd.read_excel(lp); task=None
            for _,r in df.iterrows():
                text=str(r.iloc[0])
                m=re.search(r'(?:\(|T)?(\d+)',text)
                if m: task=int(m.group(1))
                if task is None or pd.isna(r.get('Trial ID')):continue
                try: labels[(sub,task,int(r['Trial ID']))]=((float(r['Fall_onset_frame'])-1)/100.,(float(r['Fall_impact_frame'])-1)/100.)
                except: pass
    for fp in sorted(sensor.rglob('*.csv')) if sensor else []:
        m=re.search(r'S(\d+)T(\d+)R(\d+)\.csv$',fp.name)
        if not m:continue
        sub=f'SA{int(m.group(1)):02d}'; task=int(m.group(2)); rep=int(m.group(3)); df=pd.read_csv(fp)
        try:data=df[[col(df,['AccX']),col(df,['AccY']),col(df,['AccZ']),col(df,['GyrX']),col(df,['GyrY']),col(df,['GyrZ'])]].to_numpy(np.float32)
        except Exception:continue
        kind='fall' if 20<=task<=34 else 'adl'; onset,impact=labels.get((sub,task,rep),(-1,-1))
        if kind=='fall' and impact<0:impact=float(np.argmax(np.linalg.norm(data[:,:3],axis=1))/100.)
        q=win(data,100.,sub,kind,fp.stem,'KFall',impact,'waist')
        if q:parts.append(q)
    out=pack(parts); print('KFall windows:',len(out['X'])); _REAL['kf']=out; return out

def real_build_windows(name,placement='waist',**kwargs):
    if name=='SisFall':return load_sisfall()
    if name=='KFall':return load_kfall()
    if name=='UMAFall':return load_umafall()
    if name=='FallAllD':return load_fallalld()
    # Let the notebook's existing V2 adapter handle SisFall/KFall when available.
    return prep._original_build_windows(name,placement=placement,**kwargs) if hasattr(prep,'_original_build_windows') else prep.build_windows(name,placement=placement,**kwargs)
if not hasattr(prep,'_original_build_windows'):prep._original_build_windows=prep.build_windows
prep.build_windows=real_build_windows
print('Adapter installed. UMAFall:',len(load_umafall()['X']),'FallAllD:',len(load_fallalld()['X']))


In [ ]:
# CELL 4 — patch E2 and safe stage runner
p = WORK / 'src' / 'e2.py'
s = p.read_text(encoding='utf-8')
needle = '        fold_row = {\"fold\": fid, \"test\": test_name}\n        for adapter in adapters:\n'
rep = ('        if len(X_te_ev) == 0 or len(y_te_ev) == 0:\n'
       '            print(f\"[E2] SKIP fold {fid}: {test_name} has 0 evaluation windows\", flush=True)\n'
       '            continue\n\n'
       '        fold_row = {\"fold\": fid, \"test\": test_name}\n'
       '        for adapter in adapters:\n')
if needle in s:
    p.write_text(s.replace(needle, rep, 1), encoding='utf-8')
    print('E2 guard installed')
elif '0 evaluation windows' in s:
    print('E2 guard already installed')
else:
    raise RuntimeError('Expected e2.py block not found')

def invoke(fn, **kw):
    sig = inspect.signature(fn)
    if any(x.kind == inspect.Parameter.VAR_KEYWORD for x in sig.parameters.values()):
        return fn(**kw)
    return fn(**{k:v for k,v in kw.items() if k in sig.parameters})

def run_stage(module, **kw):
    print('\nSTART', module)
    try:
        m = importlib.import_module(module)
        result = invoke(m.run, **kw)
        print('DONE', module)
        return result
    except Exception as exc:
        print('STAGE FAILED BUT NOTEBOOK CONTINUES:', module, repr(exc))
        traceback.print_exc()
        return None


In [ ]:
# CELL 5 — run complete pipeline safely
results={}
if RUN_E1: results['e1']=run_stage('src.e1',epochs=EPOCHS,patience=PATIENCE,loso=False,verbose=1)
if RUN_E2: results['e2']=run_stage('src.e2',adapters=['none'],epochs=EPOCHS,patience=PATIENCE,verbose=1)
if RUN_E3: results['e3']=run_stage('src.e3',epochs=EPOCHS,patience=PATIENCE,loso=False,verbose=1)
if RUN_E4: results['e4']=run_stage('src.e4',epochs=EPOCHS,patience=PATIENCE,pretrained=False,verbose=1)
if RUN_E5: results['e5']=run_stage('src.e5',epochs=EPOCHS,patience=PATIENCE,verbose=1)
print('Pipeline finished. Some invalid stages may be reported above; no exception is re-raised.')


In [ ]:
# CELL 6 — package everything that was successfully produced
out=WORK/'output'; package=Path('/kaggle/working/fall_end_to_end_outputs')
if package.exists():shutil.rmtree(package)
package.mkdir(parents=True)
for name in ['output','figures','models','firmware']:
    q=WORK/name
    if q.exists():shutil.copytree(q,package/name)
shutil.copy2(WORK/'config.py',package/'config.py')
manifest={'data_mode':'ALL_REAL_DATASETS','roots':{k:str(v) for k,v in roots.items()},'final_run':FINAL_RUN,'epochs':EPOCHS,'umafall_windows':int(len(_REAL.get('uma',{}).get('X',[]))),'fallalld_windows':int(len(_REAL.get('fall',{}).get('X',[])))}
(package/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive('/kaggle/working/fall_end_to_end_outputs','zip',package)
print(json.dumps(manifest,indent=2));print('DOWNLOAD:',archive)
